# Bloodflow: does aggregation turn Poiseuille flow into plug flow?

A toy model: rigid disks in a 2D channel under Poiseuille flow. Turn on a
short-range (Morse) attraction and the disks aggregate into clusters that
move as a plug, carrying slow near-wall particles along — cutting the long
tail of transit times.

This notebook runs entirely in your browser (JupyterLite + Pyodide).
The two runs below take about a minute.

In [ ]:
import numpy as np
from dataclasses import replace
from bloodflow import Params, cluster_stats, simulate, transit_summary

base = Params(
    N=60, a=1.0, H=16.0, L=40.0, eta=1.0, Umax=1.0,
    kappa=2.0, eps_rep=1.0, kT=0.0, dt=0.02, seed=1,
)
results = {}
for eps in (0.0, 2.0):
    p = replace(base, eps=eps)
    res = simulate(p, n_steps=8000, equil_steps=500)
    s = transit_summary(res.transit, res.wall_time)
    c = cluster_stats(res.final_pos, cutoff=2.0 * p.a + 0.5 / p.kappa)
    results[eps] = (res, s, c)
    print(f"eps={eps}: mean={s['mean']:.1f}  median={s['median']:.1f}  "
          f"p95={s['p95']:.1f}  wall residence={s['mean_wall_residence']:.1f}  "
          f"mean cluster={c['mean_cluster_size']:.1f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
for eps, color in ((0.0, "tab:blue"), (2.0, "tab:orange")):
    res, s, c = results[eps]
    done = res.transit[~np.isnan(res.transit)]
    ax.hist(done, bins=25, alpha=0.6, color=color,
            label=f"eps={eps} (p95={s['p95']:.0f})")
ax.set_xlabel("transit time")
ax.set_ylabel("particles")
ax.legend()
ax.set_title("Transit-time distribution: no attraction vs aggregation")
plt.show()

## Takeaway

With attraction on (eps = 2), the long tail of slow transits shrinks
(p95 138 → 75), near-wall residence collapses (35 → 4), and clusters grow
(mean size 1.5 → 15). The aggregates move as a plug.

See also: the [pulsatile-flow video](videos/bloodflow-pulsatile.mp4)
(heartbeat version — the plug breathes but survives) and the
[full write-up](https://github.com/marcocaggioni/bloodflow/blob/main/README.md).